# CohortMCP -- Week 11 API Notebook

**Applied GenAI & Agentic AI Engineering Course - Week 11**

CohortMCP is a standalone MCP server (`cohort_lookup`, `incident_history`) with a browser-facing demo harness on top: direct tool calls, an LLM tool-pick endpoint, and a tool-pick accuracy eval. This notebook walks every endpoint with `curl` and Python `requests` side-by-side, then - in Section 12 - drops the HTTP harness entirely and drives the server with the **real MCP client**, the same one Claude Desktop uses.

The same server also serves the **browser UI** at `{BASE}/` and the rendered **README** at `{BASE}/readme` -- open them in a browser alongside this notebook.

---
### Before you start
1. Server running: `uvicorn app.main:app --reload`
2. `.env` filled in with `OPENAI_API_KEY` (only needed for `/eval/pick` and `/eval/run` -- the tool endpoints need no key)
3. Run the **Setup** cell below once.

> **Windows note:** All curl cells use `%%cmd` with `\"` to escape inner quotes. Single quotes are not supported in Windows cmd.
>
> **Not on localhost:8000?** Nothing below is hardcoded. Set `COHORTMCP_BASE_URL` before launching Jupyter -- a different port, a LAN address, or a tunnel URL -- and every cell follows. The Python cells read `BASE`; the `%%cmd` curl cells read `%BASE%`, which the Setup cell exports into the environment for them. If a curl cell prints a literal `%BASE%`, you skipped Setup.

In [1]:
# Setup - run this cell first
import requests, json, os

# Override to point this notebook at another host - a different port, a LAN address,
# or a tunnel:
#   PowerShell:  $env:COHORTMCP_BASE_URL = "https://xxx.trycloudflare.com"
#   bash / zsh:  export COHORTMCP_BASE_URL=https://xxx.trycloudflare.com
BASE = os.environ.get('COHORTMCP_BASE_URL', 'http://localhost:8000').rstrip('/')

# The %%cmd curl cells cannot interpolate Python variables, but a cmd subprocess
# inherits this environment - so they read %BASE% and follow the line above.
os.environ['BASE'] = BASE

# Natural-language request used across the tool-pick examples below.
DEMO_NOTES = 'Which student is handling escalation routing for the cohort?'

print('Setup complete.')
print('BASE:', BASE)

Setup complete.
BASE: http://localhost:8000


---
## 1 . Health Check -- `GET /health`
Confirms the server is alive and shows which model is loaded.

> This section is identical across all weeks. Do not modify it.

In [2]:
%%cmd
curl -s %BASE%/health

Microsoft Windows [Version 10.0.26200.8973]
(c) Microsoft Corporation. All rights reserved.

week 11>curl -s %BASE%/health
{"status":"ok","model":"gpt-5.4-nano-2026-03-17","tool_description_quality":"good","mcp_transport":"stdio"}
week 11>

In [3]:
# Health check - Python
r = requests.get(f'{BASE}/health')
print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2))

Status : 200
{
  "status": "ok",
  "model": "gpt-5.4-nano-2026-03-17",
  "tool_description_quality": "good",
  "mcp_transport": "stdio"
}


---
## 2 . List Tools -- `GET /demo/tools`

The `tools/list` shape, over HTTP -- what an MCP Inspector-style client sees: name, description, inputSchema for both tools.

Key concept: this is the exact same list a host like Claude Desktop gets when it connects to `python -m app.mcp_server` over stdio -- only the transport differs.

In [4]:
%%cmd
curl -s %BASE%/demo/tools

Microsoft Windows [Version 10.0.26200.8973]
(c) Microsoft Corporation. All rights reserved.

week 11>curl -s %BASE%/demo/tools


[{"name":"cohort_lookup","description":"Look up a cohort member's profile by student_id. Use when an orchestrator needs to know which student's service handles a given specialty, or when surfacing the cohort directory to a user. Returns {'found': false} (not an error) when student_id has no match - do not retry on 'found: false', it means the id does not exist.","inputSchema":{"properties":{"student_id":{"title":"Student Id","type":"string"}},"required":["student_id"],"title":"cohort_lookupArguments","type":"object"}},{"name":"incident_history","description":"Search recent incident history for entries whose summary contains the query text (case-insensitive substring match). Use when the user asks about past incidents, wants to see trends, or needs an incident id to reference. Returns up to 25 matches; an empty 'results' list (not an error) means nothing matched - try a broader query rather than retrying.","inputSchema":{"properties":{"query":{"title":"Query","type":"string"}},"required

In [5]:
r = requests.get(f'{BASE}/demo/tools')
print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2))

Status : 200
[
  {
    "name": "cohort_lookup",
    "description": "Look up a cohort member's profile by student_id. Use when an orchestrator needs to know which student's service handles a given specialty, or when surfacing the cohort directory to a user. Returns {'found': false} (not an error) when student_id has no match - do not retry on 'found: false', it means the id does not exist.",
    "inputSchema": {
      "properties": {
        "student_id": {
          "title": "Student Id",
          "type": "string"
        }
      },
      "required": [
        "student_id"
      ],
      "title": "cohort_lookupArguments",
      "type": "object"
    }
  },
  {
    "name": "incident_history",
    "description": "Search recent incident history for entries whose summary contains the query text (case-insensitive substring match). Use when the user asks about past incidents, wants to see trends, or needs an incident id to reference. Returns up to 25 matches; an empty 'results' list (not an 

---
## 3 . Call a Tool Directly -- `POST /demo/call`

The `tools/call` shape, over HTTP. Bypasses the model entirely -- you name the tool and pass the arguments yourself.

Request body:
```json
{ "name": "incident_history", "arguments": {"query": "payments"} }
```

Response shape:
```json
{ "success": true, "trace_id": "...", "result": {"results": [...], "total_available": 1}, "error": null }
```

In [6]:
%%cmd
curl -s -X POST %BASE%/demo/call -H "Content-Type: application/json" -d "{\"name\": \"incident_history\", \"arguments\": {\"query\": \"payments\"}}"

Microsoft Windows [Version 10.0.26200.8973]
(c) Microsoft Corporation. All rights reserved.

week 11>curl -s -X POST %BASE%/demo/call -H "Content-Type: application/json" -d "{\"name\": \"incident_history\", \"arguments\": {\"query\": \"payments\"}}"
{"success":true,"trace_id":"228b48d5a99c","result":{"results":[{"id":"INC-101","severity":"high","summary":"payments-api timeout after deploy"}],"total_available":1},"error":null}
week 11>

In [7]:
r = requests.post(f'{BASE}/demo/call', json={'name': 'incident_history', 'arguments': {'query': 'payments'}})
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print('Status :', r.status_code)
    print('\n-- Result --')
    print(json.dumps(data, indent=2))

Status : 200

-- Result --
{
  "success": true,
  "trace_id": "b250481fbe6e",
  "result": {
    "results": [
      {
        "id": "INC-101",
        "severity": "high",
        "summary": "payments-api timeout after deploy"
      }
    ],
    "total_available": 1
  },
  "error": null
}


---
## 4 . Model Picks a Tool -- `POST /eval/pick`

The pinned model (function calling, `tool_choice="required"`) decides which tool to call for a natural-language request.

Key concept: this is the "does the model actually obey the tool description" check -- a vague description (see `/eval/run` below) makes this pick less reliable.

Response shape:
```json
{ "tool": "cohort_lookup", "arguments": {"student_id": "stu_003"}, "latency_ms": 412 }
```

> Requires a real `OPENAI_API_KEY` in `.env`. Without one, expect a 502 -- see Section 9.


In [8]:
%%cmd
curl -s -X POST %BASE%/eval/pick -H "Content-Type: application/json" -d "{\"request\": \"Which student is handling escalation routing for the cohort?\"}"

Microsoft Windows [Version 10.0.26200.8973]
(c) Microsoft Corporation. All rights reserved.

week 11>curl -s -X POST %BASE%/eval/pick -H "Content-Type: application/json" -d "{\"request\": \"Which student is handling escalation routing for the cohort?\"}"
{"tool":"cohort_lookup","arguments":{"student_id":"escalation-routing"},"latency_ms":2065}
week 11>

In [9]:
r = requests.post(f'{BASE}/eval/pick', json={'request': DEMO_NOTES})
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print('Status :', r.status_code)
    print('\n-- Result --')
    print(json.dumps(data, indent=2))

Status : 200

-- Result --
{
  "tool": "cohort_lookup",
  "arguments": {
    "student_id": "escalation-routing"
  },
  "latency_ms": 1704
}


---
## 5 . Tool-Pick Eval -- `GET /eval/run`

Runs the 8-example golden set (`app/data/golden_tool_picks.json`) through `/eval/pick` and reports accuracy for the current `TOOL_DESCRIPTION_QUALITY` setting.

Key concept: this makes concrete how we measure whether the model picks the right tool, not just any tool. Flip `TOOL_DESCRIPTION_QUALITY=bad` in `.env`, restart the server, and rerun this cell to see the accuracy drop.

> Requires a real `OPENAI_API_KEY` in `.env` -- this cell makes 8 real model calls.


In [10]:
%%cmd
curl -s %BASE%/eval/run

Microsoft Windows [Version 10.0.26200.8973]
(c) Microsoft Corporation. All rights reserved.

week 11>curl -s %BASE%/eval/run
{"quality":"good","model":"gpt-5.4-nano-2026-03-17","total":8,"correct":8,"accuracy":1.0,"results":[{"id":"g1","request":"Which student handles escalation routing for the cohort?","expected_tool":"cohort_lookup","picked_tool":"cohort_lookup","correct":true},{"id":"g2","request":"Look up cohort member stu_002's profile.","expected_tool":"cohort_lookup","picked_tool":"cohort_lookup","correct":true},{"id":"g3","request":"Who is Alice and what host is her service on?","expected_tool":"cohort_lookup","picked_tool":"cohort_lookup","correct":true},{"id":"g4","request":"Have we seen any incidents related to payments timing out?","expected_tool":"incident_history","picked_tool":"incident_history","correct":true},{"id":"g5","request":"Search incident history for database replication problems.","expected_tool":"incident_history","picked_tool":"incident_history","correct":tr

In [11]:
r = requests.get(f'{BASE}/eval/run')
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print(f"Quality: {data['quality']}  Model: {data['model']}")
    print(f"Accuracy: {data['correct']}/{data['total']} = {data['accuracy']*100:.0f}%\n")
    for row in data['results']:
        mark = 'OK  ' if row['correct'] else 'MISS'
        print(f"[{mark}] expected={row['expected_tool']:<18} picked={row['picked_tool']}")

Quality: good  Model: gpt-5.4-nano-2026-03-17
Accuracy: 8/8 = 100%

[OK  ] expected=cohort_lookup      picked=cohort_lookup
[OK  ] expected=cohort_lookup      picked=cohort_lookup
[OK  ] expected=cohort_lookup      picked=cohort_lookup
[OK  ] expected=incident_history   picked=incident_history
[OK  ] expected=incident_history   picked=incident_history
[OK  ] expected=incident_history   picked=incident_history
[OK  ] expected=cohort_lookup      picked=cohort_lookup
[OK  ] expected=incident_history   picked=incident_history


---
## 6 . Full Raw Response
Shows the complete JSON as returned by the API -- useful for debugging.

In [12]:
# Full raw response dump - cohort_lookup
r = requests.post(f'{BASE}/demo/call', json={'name': 'cohort_lookup', 'arguments': {'student_id': 'stu_001'}})
print(json.dumps(r.json(), indent=2))

{
  "success": true,
  "trace_id": "aff4ab3099a4",
  "result": {
    "found": true,
    "profile": {
      "name": "Alice",
      "specialty": "knowledge-retrieval",
      "host": "localhost:8001"
    }
  },
  "error": null
}


---
## 7 . Failure Mode -- Unknown Tool (structured error, not a stack trace)

`execute_tool` never raises. An unknown tool name comes back as a normal 200 response with `success: false` and a structured error object -- the caller can branch on `error.code` without parsing a message string.

> This is a 200, not a 4xx -- the *request* to `/demo/call` is valid; it's the *tool call* that failed. Compare with Section 8, where the request itself is malformed.

In [13]:
%%cmd
curl -s -X POST %BASE%/demo/call -H "Content-Type: application/json" -d "{\"name\": \"nonexistent_tool\", \"arguments\": {}}"

Microsoft Windows [Version 10.0.26200.8973]
(c) Microsoft Corporation. All rights reserved.

week 11>curl -s -X POST %BASE%/demo/call -H "Content-Type: application/json" -d "{\"name\": \"nonexistent_tool\", \"arguments\": {}}"


{"success":false,"trace_id":"5dbba59914b3","result":null,"error":{"code":"unknown_tool","message":"Unknown tool: nonexistent_tool","retryable":false}}
week 11>

In [14]:
r = requests.post(f'{BASE}/demo/call', json={'name': 'nonexistent_tool', 'arguments': {}})
print(f'Status: {r.status_code}  (expected 200 -- success:false lives in the body, not the HTTP status)')
print(json.dumps(r.json(), indent=2))

Status: 200  (expected 200 -- success:false lives in the body, not the HTTP status)
{
  "success": false,
  "trace_id": "6ba3dc204743",
  "result": null,
  "error": {
    "code": "unknown_tool",
    "message": "Unknown tool: nonexistent_tool",
    "retryable": false
  }
}


---
## 8 . Failure Mode -- Missing Required Argument (bad_arguments)

`incident_history` requires `query`. Omitting it fails FastMCP's own argument validation before the tool ever runs (a pydantic ValidationError chained onto the SDK's ToolError), which `execute_tool` recognizes and reports as `error.code == "bad_arguments"`, `retryable: false` -- a schema mismatch, not a transient failure worth retrying.

In [15]:
%%cmd
curl -s -X POST %BASE%/demo/call -H "Content-Type: application/json" -d "{\"name\": \"incident_history\", \"arguments\": {}}"

Microsoft Windows [Version 10.0.26200.8973]
(c) Microsoft Corporation. All rights reserved.

week 11>curl -s -X POST %BASE%/demo/call -H "Content-Type: application/json" -d "{\"name\": \"incident_history\", \"arguments\": {}}"
{"success":false,"trace_id":"b04c13eeb142","result":null,"error":{"code":"bad_arguments","message":"Error executing tool incident_history: 1 validation error for incident_historyArguments\nquery\n  Field required [type=missing, input_value={}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing","retryable":false}}
week 11>

In [16]:
r = requests.post(f'{BASE}/demo/call', json={'name': 'incident_history', 'arguments': {}})
print(f'Status: {r.status_code}  (expected 200 -- structured error in the body)')
print(json.dumps(r.json(), indent=2))

Status: 200  (expected 200 -- structured error in the body)
{
  "success": false,
  "trace_id": "1f4f9489cc24",
  "result": null,
  "error": {
    "code": "bad_arguments",
    "message": "Error executing tool incident_history: 1 validation error for incident_historyArguments\nquery\n  Field required [type=missing, input_value={}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing",
    "retryable": false
  }
}


---
## 9 . Failure Mode -- Upstream Model Failure on `/eval/pick` (502)

With an invalid or missing `OPENAI_API_KEY`, `pick_tool` raises inside `app/llm.py`. `main.py` catches it and returns a clean `HTTPException(502, ...)` instead of an unhandled 500 -- the browser UI's error box shows the real upstream message rather than crashing on unparsable plain text.

In [17]:
%%cmd
curl -s -X POST %BASE%/eval/pick -H "Content-Type: application/json" -d "{\"request\": \"anything\"}"

Microsoft Windows [Version 10.0.26200.8973]
(c) Microsoft Corporation. All rights reserved.

week 11>curl -s -X POST %BASE%/eval/pick -H "Content-Type: application/json" -d "{\"request\": \"anything\"}"
{"tool":"incident_history","arguments":{"query":"anything"},"latency_ms":1449}
week 11>

In [18]:
# Only fails this way if OPENAI_API_KEY in .env is missing/invalid.
r = requests.post(f'{BASE}/eval/pick', json={'request': 'anything'})
if r.status_code == 502:
    print(f'Status: {r.status_code}  (expected 502 -- upstream model call failed cleanly)')
    print(json.dumps(r.json(), indent=2))
else:
    data = r.json()
    print(f'Key is configured -- got a real pick: {data}')

Status: 502  (expected 502 -- upstream model call failed cleanly)')


---
## 10 . OpenAPI / Swagger Docs
FastAPI auto-generates interactive docs -- try endpoints live in the browser:

> This section is identical across all weeks. Do not modify it.

In [19]:
from IPython.display import display, HTML
display(HTML(f'<a href="{BASE}/docs" target="_blank" style="font-size:15px">'
             f'Open Swagger UI: {BASE}/docs</a>'))

---
## 11 . MCP's Feature Surface -- What the Spec Offers vs. What This Build Uses

MCP defines more than tools. This build is deliberately **Tools-only** - the table below
is here so "MCP" doesn't get read as a synonym for "tool calling."

| Capability | What it's for | This build |
|---|---|---|
| **Tools** | Model-invokable functions with typed inputs/outputs | Yes -- `cohort_lookup`, `incident_history` |
| **Resources** | Read-only, addressable data a host can list/fetch without a model in the loop | Not implemented |
| **Prompts** | Reusable, parameterized prompt templates the server ships | Not implemented |
| **Sampling** | Server asks the *host's* LLM to complete something on its behalf | Not implemented |
| **Roots** | Host tells the server which filesystem roots it may touch | Not implemented |
| **Elicitation** | Server pauses mid-call to request structured input from a human | Not implemented |
| **Logging / progress / cancellation / pagination** | Protocol plumbing every MCP SDK ships | Handled transparently by `FastMCP` |

Full detail in `README.md` Section 2.


---
## 12 . The Real MCP Client -- `initialize`, `tools/list`, `tools/call`

Every section above spoke **HTTP to FastAPI**. `/demo/tools` and `/demo/call` are ordinary
REST routes that call into the SDK in-process - handy for a browser, but they are not the
protocol. There is no handshake, no capability negotiation, and no MCP host can connect
to them.

This section uses `mcp.ClientSession` - the same client Claude Desktop and the MCP
Inspector use. Watch for the two things the REST routes never showed you: a negotiated
`protocolVersion` and a `capabilities` block, both agreed at `initialize` before a single
tool is listed.


In [20]:
# --- Why the helper -------------------------------------------------------------
# Jupyter's own event loop cannot run an MCP client:
#   * on Windows the kernel uses a selector loop, which cannot spawn a subprocess
#     at all - stdio would fail with NotImplementedError;
#   * the kernel's execution thread has a small stack, and the client's nesting
#     overflows it - the kernel dies with 0xC00000FD and no traceback.
# Both problems are the notebook's, not MCP's. Give the client its own thread, its
# own loop and a bigger stack, and everything below is ordinary client code.

import asyncio, os, sys, threading

def run_mcp(factory, stack_mb=32):
    box = {}
    def worker():
        if sys.platform == "win32":
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        try:
            box["result"] = asyncio.run(factory())
        except BaseException as exc:
            box["error"] = exc
    threading.stack_size(stack_mb * 1024 * 1024)
    t = threading.Thread(target=worker)
    t.start(); t.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


# --- stdio: the client starts the server itself, no port and no second terminal ---
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def drive_stdio():
    params = StdioServerParameters(command=sys.executable, args=["-m", "app.mcp_server"])
    # errlog must be a real file: the server's stderr goes to a pipe, and Jupyter's
    # sys.stderr has no fileno() to hand the child.
    with open(os.devnull, "w") as errlog:
        async with stdio_client(params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                init = await session.initialize()
                tools = await session.list_tools()
                call = await session.call_tool("cohort_lookup", {"student_id": "stu_002"})
                return init, tools, call

init, tools, call = run_mcp(drive_stdio)

print(f"server           : {init.serverInfo.name} v{init.serverInfo.version}")
print(f"protocolVersion  : {init.protocolVersion}        <- negotiated, not hardcoded")
print(f"capabilities     : {init.capabilities.model_dump(exclude_none=True)}")
print()
for t in tools.tools:
    print(f"  {t.name}({', '.join(t.inputSchema.get('required', []))})")
    print(f"    {(t.description or '').strip().splitlines()[0][:78]}")
print()
print(f"tools/call       : {call.structuredContent}")
print(f"isError          : {call.isError}")


server           : cohortmcp v1.26.0
protocolVersion  : 2025-11-25        <- negotiated, not hardcoded
capabilities     : {'experimental': {}, 'prompts': {'listChanged': False}, 'resources': {'subscribe': False, 'listChanged': False}, 'tools': {'listChanged': False}}

  cohort_lookup(student_id)
    Look up a cohort member's profile by student_id. Use when an orchestrator need
  incident_history(query)
    Search recent incident history for entries whose summary contains the query te

tools/call       : {'found': True, 'profile': {'name': 'Bob', 'specialty': 'action-execution', 'host': 'localhost:8002'}}
isError          : False


---
### 12b . Same client, same tools, other transport

`MCP_SERVER_TRANSPORT=http python -m app.mcp_server` serves the identical registrations
over Streamable HTTP. Start it in a second terminal, then run the cell below: the tool
names, schemas and results do not move - only the two lines that say how to connect.
That is the whole point of the transport switch.


In [22]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

MCP_HTTP_URL = f"http://localhost:{os.environ.get('MCP_SERVER_PORT', '8766')}/mcp"

async def drive_http():
    async with streamablehttp_client(MCP_HTTP_URL) as (read, write, _get_session_id):
        async with ClientSession(read, write) as session:
            init = await session.initialize()
            tools = await session.list_tools()
            call = await session.call_tool("incident_history", {"query": "payments"})
            return init, tools, call

try:
    init, tools, call = run_mcp(drive_http)
    print(f"connected to     : {MCP_HTTP_URL}")
    print(f"server           : {init.serverInfo.name}")
    print(f"protocolVersion  : {init.protocolVersion}")
    print(f"tools            : {[t.name for t in tools.tools]}   <- identical to stdio")
    print(f"tools/call       : {call.structuredContent}")
except Exception as exc:
    print(f"Could not reach {MCP_HTTP_URL}: {type(exc).__name__}: {exc}")
    print("Start it first:  MCP_SERVER_TRANSPORT=http python -m app.mcp_server")


connected to     : http://localhost:8766/mcp
server           : cohortmcp
protocolVersion  : 2025-11-25
tools            : ['cohort_lookup', 'incident_history']   <- identical to stdio
tools/call       : {'results': [{'id': 'INC-101', 'severity': 'high', 'summary': 'payments-api timeout after deploy'}], 'total_available': 1}
